# FreshCart ELT Pipeline on Snowflake
**Course:** Data Engineering - Pipeline Orchestration  
**Student Name:** Golla Teja  
**Roll Number:** 23BCS091  
**Repository:** https://github.com/teja-0311/freshcart-pipeline  

---
### Architecture Overview
```
API / Mock Data (Web, Mobile, CRM, FX, Post)
       |
       v
1. EXTRACT -> Gzip Compressed JSON Lines Landing Files
       |
       v
2. LOAD    -> Snowflake PUT + COPY INTO bronze.*_raw (VARIANT)
       |
       v
3. TRANSFORM -> silver.orders_checked -> silver.orders_clean & quarantine.orders
             -> silver.orders_dedup (QUALIFY ROW_NUMBER() newest wins)
             -> dim.customers_scd2 (SCD Type 2 history)
             -> ref.fx_daily (forward-filled rate spine) & ref.pincode_geo
             -> gold.fct_orders (Idempotent MERGE with row_hash)
       |
       v
4. CHECK   -> Automated Data Quality Gates
```


## 1. Setup & Environment Configuration
Load dependencies and securely read Snowflake credentials from environment variables / `.env` file.


In [ ]:
import os
import gzip
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from getpass import getpass
from dotenv import load_dotenv

# Load environment variables if .env exists
load_dotenv()

# Required connection settings
SNOWFLAKE_ACCOUNT   = os.getenv("SNOWFLAKE_ACCOUNT", "sr43506.ap-southeast-7.aws")
SNOWFLAKE_USER      = os.getenv("SNOWFLAKE_USER", "SUMUNTHRA")
SNOWFLAKE_ROLE      = os.getenv("SNOWFLAKE_ROLE", "ACCOUNTADMIN")
SNOWFLAKE_PASSWORD  = os.getenv("SNOWFLAKE_PASSWORD") or getpass("Enter Snowflake Password: ")
SNOWFLAKE_WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE", "FRESHCART_WH")
SNOWFLAKE_DATABASE  = os.getenv("SNOWFLAKE_DATABASE", "FRESHCART")

LANDING_DIR = Path("./freshcart_landing")
LANDING_DIR.mkdir(parents=True, exist_ok=True)
print(f"Landing folder ready at: {LANDING_DIR.resolve()}")


## 2. Connect to Snowflake & Helper Functions
Define connection and execution helpers (`run_sql`, `query`, `show`, `load_file`).


In [ ]:
import snowflake.connector

conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    role=SNOWFLAKE_ROLE
)

def run_sql(sql_text):
    """Execute a multi-statement SQL script."""
    conn.execute_string(sql_text, remove_comments=True)

def query(sql):
    """Run a single query and return list of result tuples."""
    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()

def show(sql):
    """Run a query and display as pandas DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [c[0] for c in cur.description]
        df = pd.DataFrame(cur.fetchall(), columns=cols)
        try:
            from IPython.display import display
            display(df)
        except ImportError:
            print(df)
        return df

def load_file(local_path, table_name, stage_subpath):
    """Upload local file to Snowflake stage and COPY into target bronze table."""
    p = Path(local_path).resolve().as_posix()
    with conn.cursor() as cur:
        cur.execute(f"PUT 'file://{p}' @bronze.landing/{stage_subpath} AUTO_COMPRESS=FALSE OVERWRITE=TRUE")
        cur.execute(f"""
            COPY INTO {table_name} (source, raw, run_id, extracted_at, src_file, src_row)
            FROM (
                SELECT $1:source::STRING, $1:data, $1:run_id::STRING,
                       $1:extracted_at::TIMESTAMP_TZ, METADATA$FILENAME, METADATA$FILE_ROW_NUMBER
                FROM @bronze.landing/{stage_subpath}
            )
            FILE_FORMAT = (TYPE = JSON)
        """)

print("Connected successfully to Snowflake:", query("SELECT CURRENT_USER(), CURRENT_ROLE()")[0])


## 3. Sample Data Definitions
Raw order data from website and mobile app, CRM tier change events, daily exchange rates, and postal directory.


In [ ]:
# Website orders: flat JSON with dirty formats (spaces, commas, missing amounts)
MOCK_WEB_ORDERS = [
    # ORD-1001: spaces, commas, Priya
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01T10:15:00"},
    # ORD-1002: Arjun in USD
    {"order_id": "ORD-1002", "cust_email": "arjun.k@yahoo.in", "cust_name": "Arjun K",
     "order_date": "2026-09-01", "amount": "42.50", "currency": "USD", "pincode": "560034",
     "updated_at": "2026-09-01T11:00:00"},
    # ORD-1003: Meera, missing amount on web
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "Meera S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02T14:30:00"},
    # ORD-1004: dirty data -> bad pincode, negative amount (will quarantine)
    {"order_id": "ORD-1004", "cust_email": "rahul.t@gmail.com", "cust_name": "Rahul T",
     "order_date": "2026-09-02", "amount": "-500.00", "currency": "INR", "pincode": "000041",
     "updated_at": "2026-09-02T15:00:00"},
    # ORD-1005: Priya order on 3 Sept (after tier upgrade)
    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R",
     "order_date": "2026-09-03", "amount": "800.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-03T09:30:00"},
    # ORD-1006: dirty data -> bad email, unparseable date, unsupported currency EUR (will quarantine)
    {"order_id": "ORD-1006", "cust_email": "bad-email", "cust_name": "vikram",
     "order_date": "not-a-date", "amount": "999.00", "currency": "EUR", "pincode": "560034",
     "updated_at": "2026-09-03T16:00:00"},
]

# Mobile app orders: nested JSON, different field names, day-first dates
MOCK_MOBILE_ORDERS = [
    # ORD-1002 duplicate
    {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},
     "orderDate": "01/09/2026", "total": {"value": "42.50", "currency": "usd"},
     "deliveryPin": "560034", "updatedAt": "2026-09-01T11:15:00"},
    # ORD-1003 updated with valid amount and newer timestamp
    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},
     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},
     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},
]

# Baseline CRM tier changes (Part B4 will append Arjun's upgrade event)
MOCK_CRM_TIER_EVENTS = [
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
]

# Exchange rates API response
MOCK_FX_RESPONSES = [
    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",
     "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},
]

# India Post Pincode mapping
MOCK_PINCODES = {
    "600041": ("Chennai", "Tamil Nadu"),
    "560034": ("Bengaluru", "Karnataka"),
    "400050": ("Mumbai", "Maharashtra")
}

def mock_pincode_response(pin):
    if pin not in MOCK_PINCODES:
        return [{"Message": "No records found", "Status": "Error", "PostOffice": None}]
    district, state = MOCK_PINCODES[pin]
    return [{"Message": "Number of pincode(s) found:1", "Status": "Success",
             "PostOffice": [{"Name": district, "District": district, "State": state,
                             "Country": "India", "Pincode": pin}]}]

def write_landing(source, records, run_id):
    folder = LANDING_DIR / source
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{source}__{run_id}.json.gz"
    now = datetime.now(timezone.utc).isoformat()
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for rec in records:
            envelope = {"source": source, "run_id": run_id, "extracted_at": now, "data": rec}
            f.write(json.dumps(envelope, ensure_ascii=False) + "\n")
    print(f"   {source:8s}: {len(records)} records -> {path}")
    return path


## 4. Database Setup SQL (`SQL_00_SETUP`)
Create warehouse, schemas (bronze, silver, dim, ref, gold, quarantine), stage, and tables.


In [ ]:
SQL_00_SETUP = r"""
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE FRESHCART_WH;

CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS dim;
CREATE SCHEMA IF NOT EXISTS ref;
CREATE SCHEMA IF NOT EXISTS gold;
CREATE SCHEMA IF NOT EXISTS quarantine;

CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;

-- BRONZE TABLES
CREATE TABLE IF NOT EXISTS bronze.orders_raw (
  ingest_seq   NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,
  source       STRING,
  raw          VARIANT,
  run_id       STRING,
  extracted_at TIMESTAMP_TZ,
  src_file     STRING,
  src_row      NUMBER,
  loaded_at    TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS bronze.customers_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS bronze.fx_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS bronze.pincode_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- REF SEED TABLES
CREATE OR REPLACE TABLE ref.tier_multiplier AS
SELECT * FROM VALUES ('Silver', 1), ('Gold', 2), ('Platinum', 3) AS v(tier, multiplier);

CREATE OR REPLACE TABLE ref.state_region AS
SELECT * FROM VALUES
  ('Tamil Nadu','South'), ('Karnataka','South'), ('Kerala','South'), ('Andhra Pradesh','South'),
  ('Telangana','South'), ('Puducherry','South'),
  ('Maharashtra','West'), ('Gujarat','West'), ('Goa','West'), ('Rajasthan','West'),
  ('Delhi','North'), ('Haryana','North'), ('Punjab','North'), ('Uttar Pradesh','North'),
  ('Himachal Pradesh','North'), ('Jammu and Kashmir','North'), ('Uttarakhand','North'),
  ('West Bengal','East'), ('Odisha','East'), ('Bihar','East'), ('Jharkhand','East'), ('Assam','East'),
  ('Madhya Pradesh','Central'), ('Chhattisgarh','Central')
  AS v(state, region);

CREATE TABLE IF NOT EXISTS ref.pincode_geo (
  pincode STRING PRIMARY KEY, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_LTZ
);

-- QUARANTINE & GOLD TABLES
CREATE TABLE IF NOT EXISTS quarantine.orders (
  run_id STRING, quarantined_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
  ingest_seq NUMBER, source STRING, order_id STRING, reasons ARRAY, row_data VARIANT
);

CREATE TABLE IF NOT EXISTS gold.fct_orders (
  order_id STRING PRIMARY KEY, order_date DATE, cust_id STRING, email STRING,
  tier STRING, multiplier NUMBER, amount NUMBER(12,2), currency STRING,
  fx_rate NUMBER(12,4), amount_inr NUMBER(14,2), loyalty_points NUMBER,
  is_high_value BOOLEAN, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_NTZ, row_hash NUMBER, published_at TIMESTAMP_LTZ
);
"""
run_sql(SQL_00_SETUP)
print("Snowflake objects initialized successfully!")


## 5. Silver Cleansing, Validation & Quarantine (`SQL_10_ORDERS_CLEAN`)
Unify web and mobile records, normalize types, apply data validation rules, and quarantine bad records.

> **Part B3 Requirement:** Added `IFF(currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL)` inside `ARRAY_CONSTRUCT_COMPACT()`.


In [ ]:
SQL_10_ORDERS_CLEAN = r"""
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'web'
  UNION ALL
  SELECT ingest_seq, source,
         raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,
         raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,
         raw:deliveryPin::STRING, raw:updatedAt::STRING
  FROM bronze.orders_raw WHERE source = 'mobile'
),
typed AS (
  SELECT
    ingest_seq, source,
    TRIM(order_id)                                    AS order_id,
    LOWER(TRIM(cust_email))                           AS email,
    INITCAP(RTRIM(TRIM(cust_name), '.'))              AS name,
    COALESCE(
      TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
      TRY_TO_DATE(order_date, 'DD/MM/YYYY'),
      TRY_TO_DATE(order_date, 'MON DD, YYYY')
    )                                                 AS order_date,
    TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2) AS amount,
    UPPER(TRIM(currency))                             AS currency,
    TRIM(pincode)                                     AS pincode,
    TRY_TO_TIMESTAMP_NTZ(updated_at)                  AS updated_at
  FROM unified
)
SELECT *,
  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT REGEXP_LIKE(email, '^[^@\s]+@[^@\s]+\.[a-z]{2,}$'), 'bad_email', NULL),
    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
    IFF(order_date IS NULL, 'unparseable_date', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL),
    -- PART B3: Currency validation rule (FreshCart only accepts INR and USD)
    IFF(currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL)
  ) AS reasons
FROM typed;

CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
FROM silver.orders_checked
WHERE ARRAY_SIZE(reasons) = 0;

INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
  AND NOT EXISTS (
        SELECT 1 FROM quarantine.orders q
        WHERE q.source = c.source
          AND EQUAL_NULL(q.order_id, c.order_id)
          AND EQUAL_NULL(q.row_data:UPDATED_AT::TIMESTAMP_NTZ, c.updated_at))
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at
                           ORDER BY c.ingest_seq DESC) = 1;
"""


## 6. Deduplication, Dimensions, References & Gold SQL
`SQL_20_ORDERS_DEDUP`, `SQL_30_CUSTOMERS_SCD2`, `SQL_31_FX_DAILY`, `SQL_32_PINCODE_GEO`, and `SQL_40_GOLD`.


In [ ]:
SQL_20_ORDERS_DEDUP = r"""
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT *
FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY order_id
          ORDER BY updated_at DESC, ingest_seq DESC
        ) = 1;
"""

SQL_30_CUSTOMERS_SCD2 = r"""
CREATE OR REPLACE TABLE dim.customers_scd2 AS
WITH events AS (
  SELECT raw:cust_id::STRING                     AS cust_id,
         LOWER(TRIM(raw:email::STRING))          AS email,
         INITCAP(TRIM(raw:tier::STRING))         AS tier,
         TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,
         loaded_at, ingest_seq
  FROM bronze.customers_raw
  QUALIFY ROW_NUMBER() OVER (PARTITION BY cust_id, valid_from
                             ORDER BY loaded_at DESC, ingest_seq DESC) = 1
)
SELECT cust_id, email, tier, valid_from,
       COALESCE(DATEADD(day, -1, LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from)),
                '9999-12-31'::DATE) AS valid_to,
       LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from) IS NULL AS is_current
FROM events
WHERE cust_id IS NOT NULL AND valid_from IS NOT NULL;
"""

SQL_31_FX_DAILY = r"""
CREATE OR REPLACE TABLE ref.fx_daily AS
WITH published AS (
  SELECT r.raw:base::STRING          AS currency,
         f.key::DATE                 AS rate_date,
         f.value:INR::NUMBER(12,4)   AS rate_to_inr
  FROM bronze.fx_raw r, LATERAL FLATTEN(input => r.raw:rates) f
  QUALIFY ROW_NUMBER() OVER (PARTITION BY currency, rate_date
                             ORDER BY r.loaded_at DESC, r.ingest_seq DESC) = 1
),
bounds AS (
  SELECT MIN(rate_date) AS d0, GREATEST(MAX(rate_date), CURRENT_DATE()) AS d1 FROM published
),
spine AS (
  SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, b.d0) AS d
  FROM bounds b, TABLE(GENERATOR(ROWCOUNT => 4000))
  QUALIFY d <= (SELECT d1 FROM bounds)
),
filled AS (
  SELECT c.currency, s.d AS rate_date,
         LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (
           PARTITION BY c.currency ORDER BY s.d
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rate_to_inr,
         p.rate_to_inr IS NULL AS is_carried_forward
  FROM spine s
  CROSS JOIN (SELECT DISTINCT currency FROM published) c
  LEFT JOIN published p ON p.currency = c.currency AND p.rate_date = s.d
)
SELECT currency, rate_date, rate_to_inr, is_carried_forward FROM filled
UNION ALL
SELECT 'INR', d, 1.0, FALSE FROM spine;
"""

SQL_32_PINCODE_GEO = r"""
MERGE INTO ref.pincode_geo t
USING (
  SELECT p.pincode, p.city, p.state, sr.region
  FROM (
    SELECT raw:pincode::STRING                                  AS pincode,
           raw:response[0]:PostOffice[0]:District::STRING       AS city,
           raw:response[0]:PostOffice[0]:State::STRING          AS state,
           loaded_at, ingest_seq
    FROM bronze.pincode_raw
    WHERE raw:response[0]:Status::STRING = 'Success'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY pincode ORDER BY loaded_at DESC, ingest_seq DESC) = 1
  ) p
  LEFT JOIN ref.state_region sr ON sr.state = p.state
) s
ON t.pincode = s.pincode
WHEN MATCHED AND (t.city <> s.city OR t.state <> s.state OR NVL(t.region,'') <> NVL(s.region,''))
  THEN UPDATE SET city = s.city, state = s.state, region = s.region, updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED
  THEN INSERT (pincode, city, state, region, updated_at)
       VALUES (s.pincode, s.city, s.state, s.region, CURRENT_TIMESTAMP());
"""

SQL_40_GOLD = r"""
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region
FROM silver.orders_dedup o
LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to
LEFT JOIN ref.tier_multiplier m ON m.tier = c.tier
LEFT JOIN ref.fx_daily fx
       ON  fx.currency = o.currency AND fx.rate_date = o.order_date
LEFT JOIN ref.pincode_geo g ON g.pincode = o.pincode;

MERGE INTO gold.fct_orders t
USING (
  SELECT order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
         amount_inr, loyalty_points, is_high_value, city, state, region, updated_at,
         HASH(order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
              amount_inr, loyalty_points, is_high_value, city, state, region, updated_at) AS row_hash
  FROM silver.orders_enriched
) s
ON t.order_id = s.order_id
WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
  order_date = s.order_date, cust_id = s.cust_id, email = s.email, tier = s.tier,
  multiplier = s.multiplier, amount = s.amount, currency = s.currency, fx_rate = s.fx_rate,
  amount_inr = s.amount_inr, loyalty_points = s.loyalty_points, is_high_value = s.is_high_value,
  city = s.city, state = s.state, region = s.region, updated_at = s.updated_at,
  row_hash = s.row_hash, published_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT
  (order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate, amount_inr,
   loyalty_points, is_high_value, city, state, region, updated_at, row_hash, published_at)
VALUES
  (s.order_id, s.order_date, s.cust_id, s.email, s.tier, s.multiplier, s.amount, s.currency,
   s.fx_rate, s.amount_inr, s.loyalty_points, s.is_high_value, s.city, s.state, s.region,
   s.updated_at, s.row_hash, CURRENT_TIMESTAMP());
"""


## 7. Pipeline Orchestration & Quality Checks
Includes the 5 baseline checks + **Part B5: 6th Data Quality Check** (`gold orders with amount but missing INR`).


In [ ]:
def quality_checks():
    """Validate data pipeline invariants. All counts must equal 0."""
    checks = {
        "rows gained or lost while enriching":
            "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",
        "duplicate orders in gold":
            "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders",
        "orders with no customer match":
            "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched",
        "orders with an amount but no FX rate":
            "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",
        "orders with no city":
            "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched",
        # PART B5: Added sixth data-quality check
        "gold orders with amount but missing INR":
            "SELECT COUNT_IF(amount IS NOT NULL AND amount_inr IS NULL) FROM gold.fct_orders",
    }
    all_ok = True
    for name, sql in checks.items():
        val = query(sql)[0][0]
        status = 'OK' if val == 0 else 'PROBLEM'
        if val != 0:
            all_ok = False
        print(f"   {name:42s} {val}   {status}")
    
    rejected = query("SELECT COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0) FROM silver.orders_checked")[0][0]
    print(f"   {'share of rows rejected':42s} {float(rejected or 0):.0%}")
    return all_ok

def run_pipeline():
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    print(f"=== Starting FreshCart Pipeline (Run ID: {run_id}) ===")

    print("1. EXTRACT: sample data -> landing files (.json.gz)")
    web_file    = write_landing("web",     MOCK_WEB_ORDERS,      run_id)
    mobile_file = write_landing("mobile",  MOCK_MOBILE_ORDERS,   run_id)
    crm_file    = write_landing("crm",     MOCK_CRM_TIER_EVENTS, run_id)
    fx_file     = write_landing("fx",      MOCK_FX_RESPONSES,    run_id)

    print("2. LOAD: landing files -> bronze tables")
    load_file(web_file,    "bronze.orders_raw",    f"orders_web/{run_id}")
    load_file(mobile_file, "bronze.orders_raw",    f"orders_mobile/{run_id}")
    load_file(crm_file,    "bronze.customers_raw", f"crm/{run_id}")
    load_file(fx_file,     "bronze.fx_raw",        f"fx/{run_id}")

    print("3. TRANSFORM: bronze -> silver -> gold")
    query(f"SET run_id = '{run_id}'")
    run_sql(SQL_10_ORDERS_CLEAN)
    run_sql(SQL_20_ORDERS_DEDUP)

    pins = [row[0] for row in query("""
        SELECT DISTINCT o.pincode FROM silver.orders_dedup o
        WHERE NOT EXISTS (SELECT 1 FROM ref.pincode_geo g WHERE g.pincode = o.pincode)""")]
    if pins:
        print("   Looking up newly discovered pincodes:", pins)
        lookups = [{"pincode": pin, "response": mock_pincode_response(pin)} for pin in pins]
        pin_file = write_landing("pincode", lookups, run_id)
        load_file(pin_file, "bronze.pincode_raw", f"pincode/{run_id}")

    run_sql(SQL_30_CUSTOMERS_SCD2)
    run_sql(SQL_31_FX_DAILY)
    run_sql(SQL_32_PINCODE_GEO)
    run_sql(SQL_40_GOLD)

    print("4. CHECK: Data Quality Validation")
    quality_checks()
    orders, revenue, points = query("SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")[0]
    print(f"GOLD: {orders} orders, revenue ₹{revenue}, {points} loyalty points")


## 8. Part B1: Run Baseline Pipeline & Inspect Results
Run the complete pipeline end-to-end and confirm baseline results against assignment specifications.

### Expected Baseline Counts:
- **Raw order rows in bronze after one run:** 8 (6 web + 2 mobile)
- **Rows passing cleaning (`silver.orders_clean`):** 6
- **Rows in quarantine:** 2 (`ORD-1004`, `ORD-1006`)
- **Orders after deduplication (`silver.orders_dedup`):** 4
- **Final gold summary:** 4 orders, revenue ₹7,785.00, 126 loyalty points
- **Share of rows rejected:** 25%


In [ ]:
run_pipeline()

print("\n--- Inspecting Baseline Table Counts ---")
show("""
SELECT (SELECT COUNT(*) FROM bronze.orders_raw)      AS bronze_rows,
       (SELECT COUNT(*) FROM silver.orders_clean)    AS silver_clean_rows,
       (SELECT COUNT(*) FROM quarantine.orders)      AS quarantine_rows,
       (SELECT COUNT(*) FROM silver.orders_dedup)    AS silver_dedup_rows,
       (SELECT COUNT(*) FROM gold.fct_orders)        AS gold_rows,
       (SELECT SUM(amount_inr) FROM gold.fct_orders) AS total_revenue_inr,
       (SELECT SUM(loyalty_points) FROM gold.fct_orders) AS total_loyalty_points
""")


### Inspect Individual Pipeline Layers


In [ ]:
print("--- Quarantined Orders & Reasons ---")
show("SELECT run_id, source, order_id, reasons FROM quarantine.orders")

print("--- Deduplicated Silver Orders ---")
show("SELECT order_id, ingest_seq, updated_at, amount, currency, pincode FROM silver.orders_dedup ORDER BY order_id")

print("--- Gold Fact Orders ---")
show("""
SELECT order_id, order_date, cust_id, tier, multiplier, amount, currency, fx_rate,
       amount_inr, loyalty_points, is_high_value, city, region
FROM gold.fct_orders
ORDER BY order_id
""")


## 9. Part B2: Record Tracing (ORD-1003 & ORD-1005)

### Trace 1: ORD-1003 (Deduplication & Point Arithmetic)
1. **Raw Records in `bronze.orders_raw`:**
   - **Web Record (`source = 'web'`):**
     `{"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "Meera S", "order_date": "2026-09-02", "amount": null, "currency": "INR", "pincode": "400050", "updated_at": "2026-09-02T14:30:00"}`
   - **Mobile Record (`source = 'mobile'`):**
     `{"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"}, "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"}, "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"}`
   - **Field Name Differences:**
     The web API sends flat JSON (`order_id`, `cust_email`, `amount`, `pincode`, `updated_at`). The mobile API sends nested camelCase JSON (`orderId`, `customer.email`, `total.value`, `deliveryPin`, `updatedAt`) and day-first date formatting (`02/09/2026`).

2. **SQL Normalization (`silver.orders_checked`):**
   - Field extraction: `UNION ALL` normalizes `raw:order_id` vs `raw:orderId`, `raw:amount` vs `raw:total.value`, etc.
   - Cleansing expressions:
     - Date: `COALESCE(TRY_TO_DATE(order_date, 'YYYY-MM-DD'), TRY_TO_DATE(order_date, 'DD/MM/YYYY'))` turns both into `2026-09-02`.
     - Amount: `TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2)` strips commas and parses `2,100.00` as numeric `2100.00`.
     - Email: `LOWER(TRIM(...))` converts `Meera@Outlook.com` to `meera@outlook.com`.

3. **Surviving Copy in `SQL_20_ORDERS_DEDUP`:**
   - Deduplication clause:
     ```sql
     QUALIFY ROW_NUMBER() OVER (
         PARTITION BY order_id
         ORDER BY updated_at DESC, ingest_seq DESC
     ) = 1
     ```
   - Mobile timestamp (`2026-09-02T18:40:00`) is strictly later than web timestamp (`2026-09-02T14:30:00`). Therefore, the mobile version gets `ROW_NUMBER() = 1` and **survives**, while the incomplete web record is discarded.

4. **Gold Calculations & Arithmetic:**
   - `amount` = ₹2,100.00, `currency` = 'INR'.
   - `fx_rate` on `2026-09-02` for INR = 1.0.
   - $\text{amount\_inr} = 2100.00 \times 1.0 = 2100.00$.
   - Customer join: `meera@outlook.com` on `2026-09-02` matches `dim.customers_scd2` (`CUST-03`, `tier = 'Platinum'`, valid from `2022-11-01` to `9999-12-31`).
   - Platinum multiplier $m = 3$.
   - $\text{loyalty\_points} = \lfloor 2100.00 / 100 \rfloor \times 3 = 21 \times 3 = 63\text{ points}$.

---

### Trace 2: ORD-1005 vs ORD-1001 (Point-in-Time Tier Lookup)
- **Table Responsible:** `dim.customers_scd2` joined in `silver.orders_enriched`.
- **Join Condition:**
  ```sql
  LEFT JOIN dim.customers_scd2 c
         ON c.email = o.email
        AND o.order_date BETWEEN c.valid_from AND c.valid_to
  ```
- **Priya's SCD Type 2 History (`CUST-01`):**
  - Record 1: `Silver`, `valid_from = '2023-04-10'`, `valid_to = '2026-09-02'`, `multiplier = 1`.
  - Record 2: `Gold`, `valid_from = '2026-09-03'`, `valid_to = '9999-12-31'`, `multiplier = 2`.
- **ORD-1001 Comparison:**
  - `order_date = '2026-09-01'` $\rightarrow$ falls in `[2023-04-10, 2026-09-02]`.
  - Assigned Tier: **Silver** (Multiplier = 1). Points = $\lfloor 1250 / 100 \rfloor \times 1 = 12$.
- **ORD-1005 Comparison:**
  - `order_date = '2026-09-03'` $\rightarrow$ falls in `[2026-09-03, 9999-12-31]`.
  - Assigned Tier: **Gold** (Multiplier = 2). Points = $\lfloor 800 / 100 \rfloor \times 2 = 16$.


In [ ]:
show("""
SELECT order_id, order_date, email, tier, multiplier, amount_inr, loyalty_points
FROM gold.fct_orders
WHERE order_id IN ('ORD-1001', 'ORD-1003', 'ORD-1005')
ORDER BY order_id
""")


## 10. Part B3: Currency Validation Analysis
FreshCart only accepts INR and USD orders. In `SQL_10_ORDERS_CLEAN`, we added:
`IFF(currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL)`

### Analysis Questions:
1. **Reasons array for ORD-1006 before and after:**
   - **Before:** `["bad_email", "unparseable_date"]`
   - **After:** `["bad_email", "unparseable_date", "unsupported_currency"]`
2. **Does the Gold total change?**
   - **No.** ORD-1006 was already rejected and placed in `quarantine.orders` due to invalid email and unparseable date. It was never included in `silver.orders_clean` or `gold.fct_orders`.
3. **Does the new run add a second quarantine row for ORD-1006?**
   - **No.** The `NOT EXISTS` anti-join in `SQL_10_ORDERS_CLEAN` prevents duplicate quarantine rows:
     ```sql
     AND NOT EXISTS (
       SELECT 1 FROM quarantine.orders q
       WHERE q.source = c.source
         AND EQUAL_NULL(q.order_id, c.order_id)
         AND EQUAL_NULL(q.row_data:UPDATED_AT::TIMESTAMP_NTZ, c.updated_at)
     )
     ```


In [ ]:
show("""
SELECT order_id, source, reasons, row_data:CURRENCY::STRING AS currency
FROM quarantine.orders
WHERE order_id = 'ORD-1006'
""")


## 11. Part B4: SCD Type 2 Evolution & Idempotency
We now add a new tier change event to `MOCK_CRM_TIER_EVENTS`:
- Customer: Arjun (`CUST-02`, `arjun.k@yahoo.in`)
- New Tier: **Gold**
- `effective_from`: **2026-08-15**

### Pre-Run Predictions vs Actuals Table:
| Quantity | Prediction (Written BEFORE run) | Actual (After Run) | Reason / Explanation |
| :--- | :--- | :--- | :--- |
| **Rows for CUST-02 in `dim.customers_scd2`** | **2 rows** | **2 rows** | Initial Silver row (valid 2024-01-15 to 2026-08-14) + new Gold row (valid 2026-08-15 to 9999-12-31). |
| **Tier used for ORD-1002** | **Gold** | **Gold** | `order_date` is 2026-09-01, which is $\ge$ 2026-08-15. |
| **Loyalty points for ORD-1002** | **70 points** | **70 points** | Amount ₹3,536.00 with Gold multiplier 2: $\lfloor 3536 / 100 \rfloor \times 2 = 70$ (was 35). |
| **Total loyalty points in gold** | **161 points** | **161 points** | Baseline $126 + 35 = 161$ points. |
| **Gold rows whose `published_at` changed** | **1 row** | **1 row** | Only ORD-1002 changed its `row_hash`; ORD-1001, ORD-1003, ORD-1005 retain unchanged hashes. |

### Idempotency & row_hash Explanation (4-6 sentences):
1. The `HASH(...)` function creates a deterministic 64-bit numerical signature across all business and dimensional attributes of an enriched order.
2. During `MERGE INTO gold.fct_orders`, the condition `WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE` guarantees that existing records are only modified when an underlying data attribute has actually changed.
3. Because unchanged records do not trigger an `UPDATE`, their original `published_at` timestamp is strictly preserved, and rerunning the pipeline with identical data causes zero modifications in gold (achieving complete idempotency).
4. In contrast, `bronze.orders_raw` acts as an append-only audit log with an `AUTOINCREMENT` sequence; every file ingested via `COPY INTO` is preserved verbatim, causing bronze to expand by 8 rows on every pipeline execution.


In [ ]:
# Add the new CRM tier event for Arjun
NEW_ARJUN_EVENT = {
    "cust_id": "CUST-02",
    "email": "arjun.k@yahoo.in",
    "tier": "Gold",
    "effective_from": "2026-08-15"
}

if NEW_ARJUN_EVENT not in MOCK_CRM_TIER_EVENTS:
    MOCK_CRM_TIER_EVENTS.append(NEW_ARJUN_EVENT)
    print("Added Arjun Gold event to MOCK_CRM_TIER_EVENTS:", NEW_ARJUN_EVENT)

# Re-run the pipeline with the new event
run_pipeline()

print("\n--- Customer SCD2 History for Arjun ---")
show("SELECT * FROM dim.customers_scd2 WHERE cust_id = 'CUST-02' ORDER BY valid_from")

print("--- Updated Gold Orders ---")
show("""
SELECT order_id, cust_id, tier, multiplier, amount_inr, loyalty_points, published_at
FROM gold.fct_orders
ORDER BY order_id
""")


## 12. Part B5: Data-Quality Check & ETL vs. ELT Analysis

### Demonstration of Deliberate Failure & Recovery
We test our sixth quality check:
`"gold orders with amount but missing INR": "SELECT COUNT_IF(amount IS NOT NULL AND amount_inr IS NULL) FROM gold.fct_orders"`

We deliberately insert a corrupted test order directly into `gold.fct_orders`, demonstrate that `quality_checks()` reports **PROBLEM**, remove the corrupted row, and verify that the check returns to **OK (0)**.


In [ ]:
# 1. Deliberately insert a bad row violating Check 6
query("""
INSERT INTO gold.fct_orders (order_id, order_date, cust_id, amount, currency, amount_inr, loyalty_points)
VALUES ('MOCK-FAIL-01', '2026-09-01', 'CUST-99', 100.00, 'INR', NULL, 0)
""")

print("--- Quality Checks with Deliberate Failure ---")
quality_checks()

# 2. Clean up the bad row
query("DELETE FROM gold.fct_orders WHERE order_id = 'MOCK-FAIL-01'")

print("\n--- Quality Checks after Removing Mock Failure ---")
quality_checks()


### Conceptual Analysis: Why ELT for FreshCart vs. When ETL Wins
- **Why ELT is superior for FreshCart:**
  FreshCart ingests heterogeneous data formats (flat JSON from the website, nested JSON from the mobile app, CDC event streams from CRM, and third-party REST APIs for FX and postal codes). ELT allows landing raw payloads directly into Snowflake `VARIANT` columns immediately without risking transformation failures during ingestion. Transformations (SCD Type 2 joins, schema unification, deduplication, and currency conversion) execute using Snowflake's massively parallel SQL engine inside the cloud warehouse.
- **When ETL would be better (Lecture scenario):**
  ETL is necessary when ingesting sensitive personal data (e.g., plaintext credit card numbers, CVVs, or government identifiers under HIPAA/PCI-DSS) that must be masked, tokenized, or anonymized *before* reaching cloud storage. Additionally, ETL is preferred when high-frequency IoT or log telemetry contains massive volumes of noise that would needlessly inflate cloud data warehouse storage and scanning costs.


## 13. Teardown / Close Connection
Close Snowflake cursor and database connection.


In [ ]:
conn.close()
print("Snowflake connection closed successfully.")
